In [ ]:
# Кластерный метод Монте-Карло — это семейство алгоритмов, где система обновляется не по одному элементу, а целыми кластерами связанных объектов.
# В физике это особенно известно по алгоритмам Вольфа и Свендсена—Ванга: сначала строятся кластеры одинаково ориентированных спинов, а потом переворачивается сразу весь кластер.
# Алгоритм Свендсена—Ванга: вся решетка разбивается на множество кластеров, и каждый кластер переворачивается с вероятностью 1/2
# Алгоритм Вольфа: строится один кластер, начиная с одного случайно выбранного спина, после чего он переворачивается целиком.
# p = 1 - exp(-2J/T) — вероятность связать соседние одинаковые спины в кластер.
#
import numpy as np
import matplotlib.pyplot as plt
from collections import deque

def neighbors(i, j, L):# Штука чтобы возвращать соседей 
    return [((i - 1) % L, j), ((i + 1) % L, j), (i, (j - 1) % L), (i, (j + 1) % L)]# верхний , нижний  , левый , правый соседи.

def swendsen_wang_step(spins, T, J=1.0):# шаг J=1.0 — константа взаимодействия, по умолчанию равна 1.
    L = spins.shape[0]
    p = 1.0 - np.exp(-2.0 * J / T)# вероятность свзяать ближайших челов в кластер !!!!

    visited = np.zeros((L, L), dtype=bool)# записывает посещение того или иного кластера 
    new_spins = spins.copy() # записываються все конфигурации 

    for i in range(L):#  все вложенный цикл поиска соседей и обьединения в кластеры
        for j in range(L):
            if visited[i, j]:
                continue

            cluster = []
            queue = deque([(i, j)])
            visited[i, j] = True
            spin_value = spins[i, j]

            while queue:
                x, y = queue.popleft()
                cluster.append((x, y))

                for nx, ny in neighbors(x, y, L):
                    if not visited[nx, ny] and spins[nx, ny] == spin_value:
                        if np.random.rand() < p:
                            visited[nx, ny] = True
                            queue.append((nx, ny))

            if np.random.rand() < 0.5:
                for x, y in cluster:
                    new_spins[x, y] *= -1

    return new_spins

def total_energy(spins, J=1.0): # Считаем полную энергию системы 
    L = spins.shape[0]
    E = 0
    for i in range(L):
        for j in range(L):
            E -= J * spins[i, j] * (spins[(i + 1) % L, j] + spins[i, (j + 1) % L])
    return E

# Параметры
L = 32
T = 2.2
steps = 50

# Начальная конфиг
spins = np.random.choice([-1, 1], size=(L, L))

energies = []

for _ in range(steps):
    spins = swendsen_wang_step(spins, T)
    energies.append(total_energy(spins) / (L * L))

# картинка графика
plt.figure(figsize=(6, 6))
plt.imshow(spins, cmap='coolwarm', interpolation='nearest')
plt.title('Изинг-модель после алгоритма Свендсена—Ванга')
plt.colorbar(label='Спин')
plt.show()

# Энергия и шаг 
plt.figure(figsize=(7, 4))
plt.plot(energies)
plt.title('Энергия системы по шагам')
plt.xlabel('Шаг')
plt.ylabel('Энергия на сайт')
plt.grid(True)
plt.show()


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from collections import deque

# --- 1. Физика: Алгоритм Свендсена—Ванга ---

def neighbors(i, j, L):
    return [((i - 1) % L, j), ((i + 1) % L, j), (i, (j - 1) % L), (i, (j + 1) % L)]

def swendsen_wang_step(spins, T, J=1.0):
    L = spins.shape[0]
    p = 1.0 - np.exp(-2.0 * J / T)
    visited = np.zeros((L, L), dtype=bool)
    new_spins = spins.copy()

    for i in range(L):
        for j in range(L):
            if visited[i, j]: continue
            cluster = []
            queue = deque([(i, j)])
            visited[i, j] = True
            spin_value = spins[i, j]

            while queue:
                x, y = queue.popleft()
                cluster.append((x, y))
                for nx, ny in neighbors(x, y, L):
                    if not visited[nx, ny] and spins[nx, ny] == spin_value:
                        if np.random.rand() < p:
                            visited[nx, ny] = True
                            queue.append((nx, ny))
            if np.random.rand() < 0.5:
                for x, y in cluster:
                    new_spins[x, y] *= -1
    return new_spins

# --- 2. Геометрическая укладка под формат статьи (56х66, 2 канала) ---

def embed_to_fixed_matrix(spins, target_h=56, target_w=66):
    """
    Канал 0: Конфигурация спинов (+1/-1), Канал 1: Бинарная маска геометрии (1/0)
    """
    L = spins.shape[0]
    channels = np.zeros((2, target_h, target_w), dtype=np.float32)
    
    # Центрируем спиновую матрицу внутри кадра 56х66
    st_h = (target_h - L) // 2
    st_w = (target_w - L) // 2
    
    channels[0, st_h:st_h+L, st_w:st_w+L] = spins
    channels[1, st_h:st_h+L, st_w:st_w+L] = 1.0  # Бинарная маска присутствия узла
    return channels

# --- 3. Математика СНН-Классификатора (Вручную на NumPy) ---

def softmax(x):
    e_x = np.exp(x - np.max(x))
    return e_x / e_x.sum(axis=0)

def train_numpy_classifier(X, Y, lr=0.01, epochs=5):
    """ 
    Обучение классификатора: Свертка по 2 каналам -> Слой BatchNorm -> GAP -> Dense (2 класса)
    """
    np.random.seed(42)
    # Инициализируем веса сверточного ядра 3х3 для 2 входных каналов и 4 фильтров
    W_conv = np.random.randn(4, 2, 3, 3) * 0.1
    # Веса полносвязного слоя для 2 классов (выход GAP имеет размер 4)
    W_fc = np.random.randn(2, 4) * 0.1
    b_fc = np.zeros(2)
    
    print("\nШаг 2: Обучение архитектуры классификатора методом Backpropagation...")
    for epoch in range(epochs):
        loss_sum = 0.0
        for idx in range(len(X)):
            tensor = X[idx]  # Формат: (2, 56, 66)
            target = int(Y[idx])
            
            # 1. Сверточный слой (Conv2D): выход размерности (4, 54, 64)
            conv_out = np.zeros((4, 54, 64))
            for f in range(4):
                for c in range(2):
                    for i in range(54):
                        for j in range(64):
                            conv_out[f, i, j] += np.sum(tensor[c, i:i+3, j:j+3] * W_conv[f, c])
            
            # 2. Упрощенная пакетная нормализация (BatchNorm) + функция ReLU
            # Центрируем и активируем признаки
            conv_out = (conv_out - np.mean(conv_out)) / (np.std(conv_out) + 1e-5)
            activated = np.maximum(0, conv_out)
            
            # 3. Глобальное усреднение (Global Average Pooling — GAP): сжатие до вектора из 4 элементов
            gap_out = np.mean(activated, axis=(1, 2))
            
            # 4. Выходной слой и Softmax вероятности фаз
            logits = np.dot(W_fc, gap_out) + b_fc
            probs = softmax(logits)
            
            # Кросс-энтропийная функция потерь
            loss = -np.log(max(probs[target], 1e-15))
            loss_sum += float(loss)
            
            # 5. Обратное распространение ошибки (Backpropagation вручную)
            d_logits = probs.copy()
            d_logits[target] -= 1.0  # Градиент ошибки кросс-энтропии
            
            # Обновление параметров полносвязного слоя
            d_W_fc = np.outer(d_logits, gap_out)
            W_fc -= lr * d_W_fc
            b_fc -= lr * d_logits
            
            # Шаг градиента назад к сверточному слою через GAP
            d_gap = np.dot(W_fc.T, d_logits)
            for f in range(4):
                if d_gap[f] != 0:
                    # Распределяем ошибку обратно на фильтры
                    d_conv = np.where(conv_out[f] > 0, d_gap[f] / (54 * 64), 0.0)
                    for c in range(2):
                        for i in range(54):
                            for j in range(64):
                                W_conv[f, c] += lr * d_conv[i, j] * tensor[c, i:i+3, j:j+3] * 0.01

        print(f"Эпоха {epoch+1}/{epochs} | Средняя Кросс-Энтропия: {loss_sum/len(X):.4f}")
        
    return W_conv, W_fc, b_fc

# --- 4. Симуляция решетки и подготовка выборок ---

L = 32
temperatures = np.linspace(1.5, 3.2, 18)
samples_per_T = 40
thermalization_steps = 15

X_all, T_all, X_train, Y_train = [], [], [], []

print("Шаг 1: Симуляция решетки Изинга и формирование двухканальных масок...")
for T in temperatures:
    spins = np.random.choice([-1, 1], size=(L, L))
    for _ in range(thermalization_steps):
        spins = swendsen_wang_step(spins, T)
        
    for _ in range(samples_per_T):
        spins = swendsen_wang_step(spins, T)
        
        # Переводим в геометрический двухканальный формат 56х66 из статьи
        matrix_2ch = embed_to_fixed_matrix(spins)
        X_all.append(matrix_2ch)
        T_all.append(T)
        
        # Исключаем критическое окно для предотвращения переобучения
        if T < 2.1:
            X_train.append(matrix_2ch); Y_train.append(1)  # Низкотемпературная фаза
        elif T > 2.5:
            X_train.append(matrix_2ch); Y_train.append(0)  # Высокотемпературная фаза

# --- 5. Запуск обучения и классификации фаз ---

W_conv, W_fc, b_fc = train_numpy_classifier(X_train, Y_train, epochs=4)

print("\nШаг 3: Тестирование всей температурной шкалы...")
mean_p_high = []

for T in temperatures:
    preds_at_T = []
    for idx, t_val in enumerate(T_all):
        if t_val == T:
            tensor = X_all[idx]
            
            # Прямой проход (Forward pass) через обученные фильтры
            conv_out = np.zeros((4, 54, 64))
            for f in range(4):
                for c in range(2):
                    for i in range(54):
                        for j in range(64):
                            conv_out[f, i, j] += np.sum(tensor[c, i:i+3, j:j+3] * W_conv[f, c])
            
            conv_out = (conv_out - np.mean(conv_out)) / (np.std(conv_out) + 1e-5)
            activated = np.maximum(0, conv_out)
            gap_out = np.mean(activated, axis=(1, 2))
            
            logits = np.dot(W_fc, gap_out) + b_fc
            probs = softmax(logits)
            
            # Извлекаем вероятность высокотемпературной фазы (индекс 0)
            preds_at_T.append(probs[0])
            
    mean_p_high.append(np.mean(preds_at_T))

mean_p_high = np.array(mean_p_high)
mean_p_low = 1.0 - mean_p_high

# Поиск критической точки по условию P_high = 0.5 из статьи
T_c_pred = temperatures[np.argmin(np.abs(mean_p_high - 0.5))]
print(f"\n[УСПЕХ] Найденная точка фазового перехода T_c ≈ {T_c_pred:.3f}")
print("Теоретический точный предел Онзагера: 2.269")

# --- 6. Построение графика инверсии фаз (Аналог Рис. 5 из статьи) ---

plt.figure(figsize=(8, 5))
plt.plot(temperatures, mean_p_high, 'o-', color='red', linewidth=2, label='$P_{high}(T)$ (Парафаза)')
plt.plot(temperatures, mean_p_low, 'o-', color='blue', linewidth=2, label='$1 - P_{high}(T)$ (Феррофаза)')
plt.axvline(x=2.269, color='black', linestyle='--', label='Теория Онзагера (2.269)')
plt.axvline(x=T_c_pred, color='darkorange', linestyle=':', linewidth=2.5, label=f'Предсказание СНН ({T_c_pred:.2f})')

plt.title('Единый классификатор фазовых состояний (NumPy-реализация)')
plt.xlabel('Температура (T)')
plt.ylabel('Апостериорная вероятность фазы')
plt.grid(True, linestyle=':')
plt.legend()
plt.show()


In [1]:
import torch

a =torch.tensor([1, 2, 3,])
b = torch.tensor([1.2, 3.6 ,2.4])

print(a)
print(b)

tensor([1, 2, 3])
tensor([1.2000, 3.6000, 2.4000])


In [1]:
import torch.nn as nn

class MyNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc1 = nn.Linear(784, 128)  # полносвязный слой
        self.fc2 = nn.Linear(128, 10)
        self.relu = nn.ReLU()

    def forward(self, x):
        x = self.relu(self.fc1(x))
        x = self.fc2(x)
        return x

model = MyNet()

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset


X = torch.randn(1000, 784)
y = torch.randint(0, 10, (1000,))
dataset = TensorDataset(X, y)
loader  = DataLoader(dataset, batch_size=32, shuffle=True)


model     = MyNet()
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)


for epoch in range(10):
    model.train()                          
    total_loss = 0

    for X_batch, y_batch in loader:
        optimizer.zero_grad()              # сбрасываем градиенты
        output = model(X_batch)            # прямой проход
        loss   = criterion(output, y_batch)# считаем ошибку
        loss.backward()                    # обратный проход 
        optimizer.step()                   # обновляем веса
        total_loss += loss.item()

    print(f"Epoch {epoch+1}, Loss: {total_loss/len(loader):.4f}")

# 4. Оценка
model.eval()
with torch.no_grad():                     
    preds = model(X[:100])
    acc   = (preds.argmax(1) == y[:100]).float().mean()
    print(f"Accuracy: {acc:.2%}")

Epoch 1, Loss: 2.3309
Epoch 2, Loss: 1.7372
Epoch 3, Loss: 1.2319
Epoch 4, Loss: 0.7109
Epoch 5, Loss: 0.3546
Epoch 6, Loss: 0.1761
Epoch 7, Loss: 0.0992
Epoch 8, Loss: 0.0641
Epoch 9, Loss: 0.0460
Epoch 10, Loss: 0.0347
Accuracy: 100.00%
